In [3]:
import numpy as np
from scipy.special import erf
from math import sqrt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler


# ---------- 1. Data ----------

X_raw = np.array([
    [1.91447084e-01, 3.81933714e-02, 6.07417811e-01, 4.14584137e-01],
    [7.58652949e-01, 5.36517738e-01, 6.56000382e-01, 3.60341553e-01],
    [4.38349873e-01, 8.04339705e-01, 2.10245266e-01, 1.51294816e-01],
    [7.06050834e-01, 5.34191961e-01, 2.64243345e-01, 4.82087549e-01],
    [8.36477993e-01, 1.93609647e-01, 6.63892697e-01, 7.85648883e-01],
    [6.83432250e-01, 1.18662642e-01, 8.29045910e-01, 5.67576606e-01],
    [5.53621480e-01, 6.67349979e-01, 3.23805819e-01, 8.14869754e-01],
    [3.52356269e-01, 3.22241532e-01, 1.16979368e-01, 4.73112522e-01],
    [1.53785706e-01, 7.29381690e-01, 4.22598437e-01, 4.43074166e-01],
    [4.63442267e-01, 6.30024510e-01, 1.07906456e-01, 9.57643899e-01],
    [6.77491148e-01, 3.58509507e-01, 4.79592224e-01, 7.28804811e-02],
    [5.83973412e-01, 1.47242646e-01, 3.48097462e-01, 4.28614651e-01],
    [3.06888719e-01, 3.16878127e-01, 6.22634481e-01, 9.53990581e-02],
    [5.11141775e-01, 8.17956997e-01, 7.28710418e-01, 1.12353623e-01],
    [4.38933376e-01, 7.74091762e-01, 3.78167086e-01, 9.33696207e-01],
    [2.24189023e-01, 8.46480490e-01, 8.79484180e-01, 8.78515684e-01],
    [7.25261723e-01, 4.79870486e-01, 8.89468426e-02, 7.59760220e-01],
    [3.55481610e-01, 6.39619367e-01, 4.17617679e-01, 1.22603840e-01],
    [1.19879226e-01, 8.62540306e-01, 6.43331326e-01, 8.49803829e-01],
    [1.26884670e-01, 1.53429621e-01, 7.70162188e-01, 1.90518105e-01],
    [9.36477000e-01, 9.62540000e-01, 9.79484000e-01, 1.05764300e+00],
    [9.99999000e-01, 9.99999000e-01, 1.00000000e-06, 1.00000000e-06],
    [9.69909000e-01, 8.32442000e-01, 2.12340000e-01, 1.81826000e-01],
    [9.41016000e-01, 8.98976000e-01, 9.44821000e-01, 8.20504000e-01]
])

y_raw = np.array([
    6.44434399e+01, 1.83013796e+01, 1.12939795e-01, 4.21089813e+00,
    2.58370525e+02, 7.84343889e+01, 5.75715369e+01, 1.09571876e+02,
    8.84799176e+00, 2.33223610e+02, 2.44230883e+01, 6.44201468e+01,
    6.34767158e+01, 7.97291299e+01, 3.55806818e+02, 1.08885962e+03,
    2.88667516e+01, 4.51815703e+01, 4.31612757e+02, 9.97233189e+00,
    7.71337361e+03, 1.61662575e+03, 5.63309324e+02, 3.46190426e+03
]).reshape(-1, 1)


# ---------- 2. SciPy-based normal CDF (vectorised) ----------

def normal_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))


# ---------- 3. Neural Network Surrogate ----------

class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout_p=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.net(x)


def train_model(model, X_train, y_train, epochs=1500, batch_size=16, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    loader = DataLoader(TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32)
    ), batch_size=batch_size, shuffle=True)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

    return model


def mc_dropout_predict(model, X, n_samples=200):
    device = next(model.parameters()).device
    model.train()
    X_t = torch.tensor(X, dtype=torch.float32).to(device)

    preds = []
    with torch.no_grad():
        for _ in range(n_samples):
            preds.append(model(X_t).cpu().numpy().reshape(-1))

    preds = np.vstack(preds)
    return preds.mean(axis=0), preds.std(axis=0) + 1e-9


# ---------- 4. Probability of Improvement ----------

def probability_of_improvement(mu, sigma, best_y, xi=0.01):
    z = (mu - (best_y + xi)) / sigma
    pi = normal_cdf(z)
    pi[sigma < 1e-9] = 0.0
    return pi


# ---------- 5. Next Query Suggestion ----------

def propose_next_point(model, x_scaler, y_scaler, X_raw, y_raw, n_candidates=20000):

    # Current best
    best_idx = np.argmax(y_raw)
    best_x = X_raw[best_idx]
    best_y = float(y_raw[best_idx])

    # ---------- Enforce bounds [0, 1] ----------
    lb = np.zeros(4)
    ub = np.ones(4)

    # Sample candidates
    rng = np.random.default_rng(123)
    candidates = rng.uniform(lb, ub, size=(n_candidates, 4))

    # Predict
    candidates_scaled = x_scaler.transform(candidates)
    mu_s, std_s = mc_dropout_predict(model, candidates_scaled)

    mu = y_scaler.inverse_transform(mu_s.reshape(-1, 1)).reshape(-1)
    std = std_s * y_scaler.scale_[0]

    # Acquisition
    pi = probability_of_improvement(mu, std, best_y)

    idx = np.argmax(pi)

    return {
        "x_best": best_x,
        "y_best": best_y,
        "x_next": candidates[idx],
        "mu_next": mu[idx],
        "std_next": std[idx],
        "pi_next": pi[idx],
    }


# ---------- 6. Main ----------

def main():
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X = x_scaler.fit_transform(X_raw)
    y = y_scaler.fit_transform(y_raw)

    model = MLPRegressor(4)
    model = train_model(model, X, y)

    r = propose_next_point(model, x_scaler, y_scaler, X_raw, y_raw)

    print("\n===== Current Best =====")
    print("x_best:", r["x_best"])
    print("y_best:", r["y_best"])

    print("\n===== Proposed Next Query (bounds 0–1) =====")
    print("x_next:", r["x_next"])
    print(f"Predicted mean: {r['mu_next']:.4f}")
    print(f"Predicted std:  {r['std_next']:.4f}")
    print(f"Probability of Improvement: {r['pi_next']:.4f} ({r['pi_next']*100:.2f}%)")


if __name__ == "__main__":
    main()


C:\Users\veeja\AppData\Local\Temp\ipykernel_73960\44711881.py:122: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  best_y = float(y_raw[best_idx])



===== Current Best =====
x_best: [0.936477 0.96254  0.979484 1.057643]
y_best: 7713.37361

===== Proposed Next Query (bounds 0–1) =====
x_next: [0.797068 0.977185 0.882682 0.998829]
Predicted mean: 5931.3340
Predicted std:  567.1296
Probability of Improvement: 0.0008 (0.08%)
